# Vordefinierte Ablation der News-Features

Dieses Notebook untersucht, welche Gruppen der Nachrichtenmerkmale zusätzlich zu den numerischen Merkmalen zur Vorhersageleistung beitragen. Dafür sollen verschiedene Merkmalskombinationen unter denselben Modellbedingungen verglichen werden.

Die Analyse dient dem Verständnis der Nachrichtenmerkmale. Auf Grundlage der Testergebnisse wird kein neues finales Modell ausgewählt.


In [22]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

from buli_news.modeling.combined import load_combined_classification_data
from buli_news.modeling.evaluation import (
    SELECTED_LOGISTIC_C,
    SELECTED_NUMERICAL_FEATURE_COLUMNS,
    evaluate_logistic_classification_data,
    select_classification_features,
    validate_numerical_logistic_selection_report,
)
from buli_news.news.features import NEWS_FEATURE_COLUMNS

pd.set_option("display.float_format", lambda value: f"{value:.3f}")

SEASON = 2025


## 1. Eingabedateien

Einlesen der numerischen Merkmale und der aggregierten Nachrichtenmerkmale aus der Pipeline.

Die gemeinsame Modellierungsfunktion verbindet beide Tabellen und stellt die festen Trainings- und Testdaten bereit. Der Selection-Report wird mit der eingefrorenen numerischen Konfiguration abgeglichen.


In [23]:
NUMERICAL_FEATURES_PATH = Path(f"../data/processed/numerical_features_{SEASON}.csv")
NEWS_FEATURES_PATH = Path(f"../data/processed/news_features_{SEASON}.csv")

SELECTION_REPORT_PATH = Path(f"../outputs/modeling/{SEASON}/numerical/logistic_regression/selection/report.json")
validate_numerical_logistic_selection_report(SELECTION_REPORT_PATH, SEASON)

combined_data = load_combined_classification_data(
    numerical_features_path=NUMERICAL_FEATURES_PATH,
    news_features_path=NEWS_FEATURES_PATH,
    season=SEASON,
).data

print(f"Trainingsspiele: {len(combined_data.X_train)}")
print(f"Testspiele: {len(combined_data.X_test)}")


Trainingsspiele: 243
Testspiele: 63


## 2. Gesamtgüte der Pipeline-Modelle

Die Tabelle zeigt die gespeicherten Ergebnisse der vier Pipeline-Modelle auf denselben 63 Testspielen als Referenz für die folgenden Vergleiche. Die Kennzahlen werden direkt aus den Modellreports eingelesen.


In [24]:
MODEL_REPORT_PATHS = {
    "ZeroR baseline": Path(f"../outputs/modeling/{SEASON}/numerical/dummy/evaluation.json"),
    "Selected numerical model": Path(f"../outputs/modeling/{SEASON}/numerical/logistic_regression/final/evaluation.json"),
    "News-only diagnostic model": Path(f"../outputs/modeling/{SEASON}/news/logistic_regression/diagnostic/evaluation.json"),
    "Combined model": Path(f"../outputs/modeling/{SEASON}/combined/logistic_regression/final/evaluation.json"),
}

model_metrics = {}
for model_name, report_path in MODEL_REPORT_PATHS.items():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    metrics = report["metrics"]
    model_metrics[model_name] = {
        "Log Loss": metrics["log_loss"],
        "Brier Score": metrics["multiclass_brier_score"],
        "Accuracy": metrics["accuracy"],
        "Macro-F1": metrics["macro_f1"],
    }

global_metrics = pd.DataFrame.from_dict(model_metrics, orient="index")
display(global_metrics)


,Log Loss,Brier Score,Accuracy,Macro-F1
ZeroR baseline,1.096,0.666,0.381,0.184
Selected numerical model,1.061,0.636,0.492,0.367
News-only diagnostic model,1.080,0.660,0.397,0.250
Combined model,1.053,0.630,0.476,0.360


## 3. Mittelwerte und Mention Shares getrennt untersuchen

Beide Varianten behalten die 31 numerischen Merkmale. Einmal kommen nur die acht News-Mittelwerte hinzu, einmal nur die acht Mention Shares. So untersuchen wir getrennt den Zusatznutzen der Bewertungen und der Erwähnungsanteile.

Beide Modelle werden mit der bestehenden Standardisierung und logistischen Regression bei `C=0.01` auf den 243 Trainingsspielen neu trainiert und auf denselben 63 Testspielen ausgewertet.

### Numerische Merkmale + News-Mittelwerte


In [25]:
mean_columns = tuple(
    column for column in NEWS_FEATURE_COLUMNS if column.endswith("_mean_rating")
)
mean_data = select_classification_features(
    data=combined_data,
    feature_columns=(*SELECTED_NUMERICAL_FEATURE_COLUMNS, *mean_columns),
)
mean_evaluation = evaluate_logistic_classification_data(
    data=mean_data,
    season=SEASON,
    experiment="news_ablation_mean_only",
    C=SELECTED_LOGISTIC_C,
    input_reference={
        "numerical_features": NUMERICAL_FEATURES_PATH,
        "news_features": NEWS_FEATURES_PATH,
    },
)

mean_metrics = mean_evaluation.report["metrics"]
display(pd.DataFrame({
    "Log Loss": mean_metrics["log_loss"],
    "Brier Score": mean_metrics["multiclass_brier_score"],
    "Accuracy": mean_metrics["accuracy"],
    "Macro-F1": mean_metrics["macro_f1"],
}, index=["Selected numerical model + ratings only"]))


,Log Loss,Brier Score,Accuracy,Macro-F1
Selected numerical model + ratings only,1.068,0.637,0.492,0.366


### Numerische Merkmale + Mention Shares

In [26]:
mention_columns = tuple(
    column for column in NEWS_FEATURE_COLUMNS if column.endswith("_mention_share")
)
mention_data = select_classification_features(
    data=combined_data,
    feature_columns=(*SELECTED_NUMERICAL_FEATURE_COLUMNS, *mention_columns),
)
mention_evaluation = evaluate_logistic_classification_data(
    data=mention_data,
    season=SEASON,
    experiment="news_ablation_mention_only",
    C=SELECTED_LOGISTIC_C,
    input_reference={
        "numerical_features": NUMERICAL_FEATURES_PATH,
        "news_features": NEWS_FEATURES_PATH,
    },
)

mention_metrics = mention_evaluation.report["metrics"]
display(pd.DataFrame({
    "Log Loss": mention_metrics["log_loss"],
    "Brier Score": mention_metrics["multiclass_brier_score"],
    "Accuracy": mention_metrics["accuracy"],
    "Macro-F1": mention_metrics["macro_f1"],
}, index=["Selected numerical model + mentions only"]))


,Log Loss,Brier Score,Accuracy,Macro-F1
Selected numerical model + mentions only,1.049,0.630,0.492,0.374


## 4. Einzelne Nachrichtenindikatoren untersuchen

Jede der vier Varianten ergänzt die 31 numerischen Merkmale um genau einen Nachrichtenindikator: jeweils Mittelwert und Mention Share für Heim- und Auswärtsteam. Damit verwendet jedes Modell 35 Merkmale.


In [27]:
INDICATOR_LABELS = {
    "sporting_form": "Sportliche Form",
    "personnel_situation": "Personalsituation",
    "physical_readiness": "Körperliche Verfassung",
    "confidence_and_motivation": "Selbstvertrauen und Motivation",
}

indicator_metrics = {}
for indicator, label in INDICATOR_LABELS.items():
    indicator_columns = tuple(
        column for column in NEWS_FEATURE_COLUMNS if f"_news_{indicator}_" in column
    )
    indicator_data = select_classification_features(
        data=combined_data,
        feature_columns=(*SELECTED_NUMERICAL_FEATURE_COLUMNS, *indicator_columns),
    )
    indicator_evaluation = evaluate_logistic_classification_data(
        data=indicator_data,
        season=SEASON,
        experiment=f"news_ablation_{indicator}_only",
        C=SELECTED_LOGISTIC_C,
        input_reference={
            "numerical_features": NUMERICAL_FEATURES_PATH,
            "news_features": NEWS_FEATURES_PATH,
        },
    )
    metrics = indicator_evaluation.report["metrics"]
    indicator_metrics[f"Selected numerical model + {label}"] = {
        "Log Loss": metrics["log_loss"],
        "Brier Score": metrics["multiclass_brier_score"],
        "Accuracy": metrics["accuracy"],
        "Macro-F1": metrics["macro_f1"],
    }

indicator_metric_table = pd.DataFrame.from_dict(indicator_metrics, orient="index")
display(indicator_metric_table)


,Log Loss,Brier Score,Accuracy,Macro-F1
Selected numerical model + Sportliche Form,1.051,0.632,0.492,0.370
Selected numerical model + Personalsituation,1.072,0.642,0.492,0.367
Selected numerical model + Körperliche Verfassung,1.073,0.641,0.476,0.355
Selected numerical model + Selbstvertrauen und Motivation,1.043,0.626,0.508,0.380


## 5. Gesamtvergleich aller Modelle

Die Tabelle fasst die vier Pipeline-Modelle und die sechs Ablationsvarianten zusammen, aufsteigend nach Log Loss sortiert.


In [28]:
mean_mention_metrics = pd.DataFrame.from_dict({
    "Selected numerical model + ratings only": mean_metrics,
    "Selected numerical model + mentions only": mention_metrics,
}, orient="index").rename(columns={
    "log_loss": "Log Loss",
    "multiclass_brier_score": "Brier Score",
    "accuracy": "Accuracy",
    "macro_f1": "Macro-F1",
})

all_model_metrics = pd.concat([
    global_metrics,
    mean_mention_metrics[global_metrics.columns],
    indicator_metric_table,
]).sort_values("Log Loss", kind="stable")

display(all_model_metrics)


,Log Loss,Brier Score,Accuracy,Macro-F1
Selected numerical model + Selbstvertrauen und Motivation,1.043,0.626,0.508,0.380
Selected numerical model + mentions only,1.049,0.630,0.492,0.374
Selected numerical model + Sportliche Form,1.051,0.632,0.492,0.370
Combined model,1.053,0.630,0.476,0.360
Selected numerical model,1.061,0.636,0.492,0.367
Selected numerical model + ratings only,1.068,0.637,0.492,0.366
Selected numerical model + Personalsituation,1.072,0.642,0.492,0.367
Selected numerical model + Körperliche Verfassung,1.073,0.641,0.476,0.355
News-only diagnostic model,1.080,0.660,0.397,0.250
ZeroR baseline,1.096,0.666,0.381,0.184
